# Taller final — Minería de datos
## Segmentación de clientes con K-Means

**Dataset:** Mall Customers  
**Objetivo:** desarrollar un modelo de segmentación con K-Means, evaluar el número de grupos (método del codo y silueta) e interpretar los perfiles para marketing. Al final se despliega una app en Streamlit.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 5)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "Mall_Customers.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT.parent / "data" / "Mall_Customers.csv"
MODELS_DIR = (ROOT if (ROOT / "data").exists() else ROOT.parent) / "models"
MODELS_DIR.mkdir(exist_ok=True)
print("Datos:", DATA_PATH.resolve())

---
## Parte 1 — Exploración de datos

### 1. Cargar el dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Forma:", df.shape)
print("\nTipos de dato:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())
df.info()

### 2. Identificar variables relevantes

In [ ]:
print("Columnas:", list(df.columns))
print("\nDistribución de Gender:")
print(df["Gender"].value_counts())
print("\nCustomerID es un identificador único:", df["CustomerID"].nunique() == len(df))

### 3. Análisis descriptivo básico

In [ ]:
df.describe(include="all")

In [ ]:
num_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f"Boxplot — {col}")
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, vars=num_cols, hue="Gender", diag_kind="kde")
plt.suptitle("Relaciones entre variables numéricas", y=1.02)
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación de Pearson")
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    s=60,
)
plt.title("Ingreso anual vs. puntaje de gasto")
plt.show()

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

print("Detección de atípicos (regla IQR, 1.5):\n")
for col in num_cols:
    mask, lower, upper = iqr_outliers(df[col])
    print(f"{col}: límites [{lower:.1f}, {upper:.1f}] — atípicos: {mask.sum()}")
    if mask.any():
        display(df.loc[mask, ["CustomerID", "Gender"] + num_cols])

### Preguntas — Parte 1

**¿Qué variables son más adecuadas para segmentar clientes?**

`CustomerID` no se usa: es un identificador, no un atributo de comportamiento. `Gender` es categórica y, en K-Means (distancia euclidiana), no aporta una métrica continua comparable; además, el pairplot no muestra grupos claramente separados por género.

Las variables más adecuadas son **`Annual Income (k$)`** y **`Spending Score (1-100)`**: combinan capacidad de compra y disposición a gastar, que es lo que el área de marketing necesita para armar campañas. `Age` puede complementar (ciclo de vida), pero el patrón de clusters más claro se ve en el plano ingreso–gasto. Por eso el modelo se entrena con esas dos variables (enfoque clásico de este dataset).

**¿Se identifican valores atípicos?**

Con la regla IQR, **Age** y **Spending Score** no muestran atípicos. En **Annual Income** pueden aparecer uno o dos ingresos altos (cerca del techo del dataset, ~137 k$). No son errores evidentes: son clientes con mayor poder adquisitivo. No se eliminan, porque en segmentación esos puntos pueden ser precisamente un grupo de alto valor. K-Means es sensible a extremos, por eso más adelante se **estandarizan** las variables.

**¿Qué tipo de relaciones se observan entre las variables?**

- La correlación lineal entre ingreso y puntaje de gasto es **casi nula**: gana más no implica gastar más en el centro comercial.
- Edad e ingreso también se relacionan poco; edad y gasto muestran una correlación **leve y negativa** (clientes más jóvenes tienden a puntajes de gasto un poco más altos).
- En el diagrama de dispersión ingreso vs. gasto se ven **nubes o grupos naturales** (bajo ingreso/alto gasto, alto ingreso/bajo gasto, centro, etc.). Eso justifica usar clustering no supervisado en lugar de una sola recta de regresión.

---
## Parte 2 — Preprocesamiento

Se aplica `StandardScaler` (media 0, desviación 1) para que ingreso y puntaje de gasto pesen igual en la distancia euclidiana.

In [ ]:
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Media original:\n", X.mean().round(2))
print("\nDesviación original:\n", X.std().round(2))
print("\nTras StandardScaler — medias ≈ 0:", np.round(X_scaled.mean(axis=0), 4))
print("Tras StandardScaler — desv. ≈ 1:", np.round(X_scaled.std(axis=0), 4))
pd.DataFrame(X_scaled, columns=features).head()

### Pregunta — Parte 2

**¿Por qué es necesario escalar las variables en el algoritmo K-Means?**

K-Means asigna cada cliente al centroide más cercano usando **distancia euclidiana**. Si una variable tiene un rango mucho mayor que otra, **domina esa distancia**. Aquí el ingreso está en miles de dólares (~15 a 137) y el puntaje de gasto va de 1 a 100: sin escalar, el ingreso pesaría más y los grupos se formarían casi solo por dinero.

`StandardScaler` deja ambas variables en la misma escala (z-scores). Así, un cambio de una desviación típica en ingreso vale lo mismo que en gasto, que es lo que queremos para una segmentación equilibrada.

---
## Parte 3 — Método del codo

Se calcula la **inercia** (suma de distancias al cuadrado de cada punto a su centroide) para K = 1 … 10.

In [ ]:
K_range = range(1, 11)
inertias = []

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

elbow_df = pd.DataFrame({"K": list(K_range), "Inercia": inertias})
display(elbow_df)

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), inertias, marker="o")
plt.xticks(list(K_range))
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inercia (WCSS)")
plt.title("Método del codo")
plt.axvline(5, color="red", linestyle="--", alpha=0.7, label="Codo candidato (K=5)")
plt.legend()
plt.show()

### Preguntas — Parte 3

**¿Por qué valores cercanos a cero indican mejor ajuste?**

La inercia (WCSS) mide qué tan **compactos** son los grupos: si cada punto está muy cerca de su centroide, la suma de cuadrados es pequeña, cercana a cero. Un valor bajo significa que el modelo “explica” bien la nube de puntos **dentro** de cada cluster. El límite es trivial: con K igual al número de clientes la inercia sería 0, pero eso no sirve. Por eso no se busca el mínimo absoluto, sino el K a partir del cual **bajar más la inercia ya no justifica un cluster extra**.

**¿Cómo se identifica el punto óptimo de K?**

Se busca el **codo** de la curva: el K donde la inercia deja de caer de forma pronunciada y la gráfica se aplana. En este dataset ese cambio de pendiente ocurre de forma clara en **K = 5**. Pasar a 6 o 7 reduce poco la inercia y parte grupos que ya son interpretables para el negocio.

---
## Parte 4 — Coeficiente de silueta

In [ ]:
K_sil = range(2, 11)
sil_scores = []

for k in K_sil:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

sil_df = pd.DataFrame({"K": list(K_sil), "Silueta": sil_scores})
display(sil_df.style.highlight_max(subset=["Silueta"], color="lightgreen"))

plt.figure(figsize=(8, 5))
plt.plot(list(K_sil), sil_scores, marker="o", color="teal")
plt.xticks(list(K_sil))
plt.xlabel("Número de clusters (K)")
plt.ylabel("Coeficiente de silueta (promedio)")
plt.title("Silueta vs. K")
best_k_sil = list(K_sil)[int(np.argmax(sil_scores))]
plt.axvline(best_k_sil, color="red", linestyle="--", alpha=0.7, label=f"Máximo en K={best_k_sil}")
plt.legend()
plt.show()

print(f"Mejor K según silueta: {best_k_sil} (score = {max(sil_scores):.3f})")

In [ ]:
K_SELECTED = 5
km_sil_plot = KMeans(n_clusters=K_SELECTED, init="k-means++", n_init=10, random_state=42)
labels_tmp = km_sil_plot.fit_predict(X_scaled)
sample_sil = silhouette_samples(X_scaled, labels_tmp)

y_lower = 10
plt.figure(figsize=(8, 6))
for i in range(K_SELECTED):
    ith = sample_sil[labels_tmp == i]
    ith.sort()
    size = ith.shape[0]
    y_upper = y_lower + size
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, ith, alpha=0.7)
    plt.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower = y_upper + 10

plt.axvline(silhouette_score(X_scaled, labels_tmp), color="red", linestyle="--")
plt.title(f"Gráfico de silueta para K={K_SELECTED}")
plt.xlabel("Coeficiente de silueta")
plt.ylabel("Cluster")
plt.show()

### Preguntas — Parte 4

**¿Qué mide el coeficiente de silueta?**

Para cada cliente compara dos distancias: (a) qué tan cerca está de los demás de **su** cluster (cohesión) y (b) qué tan cerca está del cluster **vecino** más próximo (separación). El promedio de esos valores resume si los grupos están **compactos y bien separados**.

**¿Qué significa un valor cercano a 1?**

El punto está bien asignado: está mucho más cerca de su propio grupo que de cualquier otro. Un promedio cercano a 1 indica una partición **clara y estable**.

**¿Qué implica un valor cercano a 0 o negativo?**

- Cercano a **0**: el cliente está en la **frontera** entre dos grupos; podría pertenecer a cualquiera.
- **Negativo**: está, en promedio, más cerca de otro cluster que del suyo: **mala asignación** (solapamiento o K inadecuado).

**¿Coincide el valor óptimo de K con el método del codo? Justifique.**

Sí: el codo apunta a **K = 5** y la silueta suele **maximizarse también en K = 5** (alrededor de 0.55, que es un valor razonable en datos reales). Ambos criterios coinciden, así que se adopta **K = 5** para el modelo final: es el equilibrio entre compactación (codo) y separación (silueta), y además produce perfiles de negocio interpretables.

---
## Parte 5 — Modelo K-Means

In [ ]:
K_SELECTED = 5
model = KMeans(n_clusters=K_SELECTED, init="k-means++", n_init=10, random_state=42)
df["Cluster"] = model.fit_predict(X_scaled)

print("Clientes por cluster:")
print(df["Cluster"].value_counts().sort_index())
print("\nInercia final:", round(model.inertia_, 2))
print("Silueta final:", round(silhouette_score(X_scaled, df["Cluster"]), 3))

In [ ]:
centroids_scaled = model.cluster_centers_
centroids = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids, columns=features)
centroids_df.index.name = "Cluster"
print("Centroides (escala original):")
display(centroids_df.round(2))

print("\nPromedios por cluster (incluye edad para interpretar):")
profile = df.groupby("Cluster")[num_cols].mean().round(2)
profile["n"] = df.groupby("Cluster").size()
display(profile)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="Set2",
    s=70,
)
plt.scatter(
    centroids_df["Annual Income (k$)"],
    centroids_df["Spending Score (1-100)"],
    c="black",
    s=200,
    marker="X",
    label="Centroides",
)
plt.title(f"Clusters K-Means (K={K_SELECTED})")
plt.legend()
plt.show()

In [ ]:
joblib.dump(model, MODELS_DIR / "kmeans_model.joblib")
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
joblib.dump(features, MODELS_DIR / "features.joblib")
print("Modelo y scaler guardados en", MODELS_DIR.resolve())

### Preguntas — Parte 5

**¿Qué representa cada cluster?**

Cada cluster es un **segmento de clientes con comportamiento similar** en ingreso y gasto. El centroide es el “cliente típico” de ese grupo. Los cinco grupos que aparecen en este dataset son:

| Cluster (típico) | Ingreso | Gasto | Lectura |
|---|---|---|---|
| Alto valor | Alto | Alto | Compran mucho y pueden pagar |
| Potencial | Alto | Bajo | Tienen dinero pero casi no compran en el mall |
| Promedio | Medio | Medio | Núcleo del negocio |
| Cuidado / sensibles al precio | Bajo | Alto | Gastan mucho relativo a su ingreso |
| Bajo valor | Bajo | Bajo | Poco ingreso y poco gasto |

**¿Qué diferencias existen entre los grupos obtenidos?**

La diferencia principal **no es la edad ni el género**, sino la **combinación ingreso–gasto**:

- Hay clientes ricos que **sí** son fieles al centro comercial y otros ricos que **no** lo son (misma capacidad, distinto comportamiento).
- Hay clientes de ingreso bajo que aun así tienen **alto puntaje de gasto** (buscan promociones, o priorizan este canal).
- El grupo central es el más numeroso: ingreso y gasto moderados.

Eso impide tratar a “todos los de alto ingreso” igual: hay que mirar también el spending score.

---
## Parte 6 — Interpretación de negocio

In [ ]:
inc_med = df["Annual Income (k$)"].median()
score_med = df["Spending Score (1-100)"].median()

def etiquetar(row):
    alto_ing = row["Annual Income (k$)"] >= inc_med
    alto_gasto = row["Spending Score (1-100)"] >= score_med
    if alto_ing and alto_gasto:
        return "Clientes de alto valor"
    if alto_ing and not alto_gasto:
        return "Clientes con potencial"
    if (not alto_ing) and alto_gasto:
        return "Clientes sensibles / alto gasto"
    return "Clientes de bajo valor"

centro = centroids_df.copy()
centro["Perfil"] = centro.apply(etiquetar, axis=1)

# El cluster más cercano a la mediana en ambas variables se marca como promedio
dist_centro = np.sqrt(
    ((centro["Annual Income (k$)"] - inc_med) / df["Annual Income (k$)"].std()) ** 2
    + ((centro["Spending Score (1-100)"] - score_med) / df["Spending Score (1-100)"].std()) ** 2
)
cluster_promedio = int(dist_centro.idxmin())
centro.loc[cluster_promedio, "Perfil"] = "Clientes promedio"

mapa_perfiles = centro["Perfil"].to_dict()
df["Perfil"] = df["Cluster"].map(mapa_perfiles)

display(centro.round(2))
print("\nTamaño de cada perfil:")
print(df["Perfil"].value_counts())

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Perfil",
    s=70,
)
plt.title("Perfiles de negocio")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

resumen = (
    df.groupby("Perfil")[num_cols]
    .mean()
    .round(1)
    .assign(Clientes=df.groupby("Perfil").size())
)
display(resumen)

### Preguntas — Parte 6

**¿Qué estrategias de marketing se podrían aplicar a cada grupo?**

- **Alto valor:** programas de fidelización, VIP, lanzamientos y personal shopper. El objetivo es **retener**, no recortar precio.
- **Promedio:** cross-selling y bundles (café + cine, moda + accesorios), comunicación regular sin descuentos agresivos.
- **Bajo valor:** campañas de bajo costo (app, email), ofertas de entrada; no invertir presupuesto premium.
- **Con potencial (alto ingreso, bajo gasto):** es el grupo a **conquistar**: experiencias, marcas premium, eventos exclusivos, first-visit offers. Tienen dinero; hoy no lo gastan aquí.
- **Sensibles / alto gasto con ingreso bajo:** cupones, 2x1 y financiación; cuidar no incentivar gasto insostenible, pero sí canalizarlo a categorías de margen.

**¿Qué segmento representa mayor oportunidad de crecimiento?**

El de **clientes con potencial** (ingreso alto y gasto bajo). Ya tienen capacidad de compra; el problema es de **preferencia de canal**, no de bolsillo. Subir su spending score un poco genera más ingreso incremental que exprimir al grupo de bajo valor o dar más descuento a quienes ya son VIP. El segundo foco es **retener alto valor** (evitar fuga), que protege la base más rentable.

---
## Parte 7 — Despliegue con Streamlit

El modelo y el scaler quedaron en la carpeta `models/`. La aplicación está en `app.py`.

Desde la carpeta del proyecto, en una terminal:

```bash
pip install -r requirements.txt
streamlit run app.py
```

La app permite ver el método del codo, los clusters y **clasificar un cliente nuevo** (ingreso + puntaje de gasto).